# MAI-Voice-1: High-Fidelity Text-to-Speech

> **Model Card:** [ai.azure.com/catalog/models/MAI-Voice-1](https://ai.azure.com/catalog/models/MAI-Voice-1)

MAI-Voice-1 is Microsoft AI's first-generation speech generation model — capable of producing **60 seconds of expressive audio in under one second** on a single GPU. It powers Copilot Audio Expressions, podcast features, and is available through Azure Speech.

| Attribute | Detail |
|---|---|
| **Model type** | Text-to-Speech (TTS) |
| **Languages** | English (10+ expanding soon) |
| **Input** | Plain text · SSML (emotion/tone control) |
| **Output** | MP3 · WAV · Opus · FLAC |
| **Voice features** | Curated voice library · Voice prompting (clone from 10s clip) |
| **Regions** | Central US · Japan West · Sweden Central |
| **API** | Azure Speech REST API (`/cognitiveservices/v1`) · Speech SDK |
| **Pricing** | **\$22 / 1M characters** |
| **Powers** | Copilot Audio Expressions, Copilot podcasts |

### Key capabilities
- **High-fidelity natural voice** — human-like intonation, rhythm, and emotional range
- **Voice prompting** — provide a few-second audio clip, model clones it instantly (no fine-tuning)
- **Emotion/tone control** — shape delivery at the turn/sentence level via SSML
- **Long-form content** — stable speaker consistency across full audiobooks, lectures, podcasts

## 1. Setup

In [ ]:
%pip install -q azure-cognitiveservices-speech python-dotenv requests azure-identity

In [ ]:
import os
import io
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

import azure.cognitiveservices.speech as speechsdk

load_dotenv()

VOICE_SPEECH_KEY = os.getenv("VOICE_SPEECH_KEY") or os.getenv("SPEECH_KEY")
VOICE_SPEECH_REGION = os.getenv("VOICE_SPEECH_REGION") or os.getenv("SPEECH_REGION", "centralus")
MAI_VOICE_NAME = os.getenv("MAI_VOICE_NAME", "en-US-MAIVoice1Neural")

# Backward-compatible aliases used in later cells
SPEECH_KEY = VOICE_SPEECH_KEY
SPEECH_REGION = VOICE_SPEECH_REGION

assert VOICE_SPEECH_KEY, "Set VOICE_SPEECH_KEY (or SPEECH_KEY) in your .env file"
assert VOICE_SPEECH_REGION, "Set VOICE_SPEECH_REGION (or SPEECH_REGION) in your .env file"

# TTS REST endpoint
TTS_ENDPOINT = (
    f"https://{VOICE_SPEECH_REGION}.tts.speech.microsoft.com"
    "/cognitiveservices/v1"
 )

print(f"✅ TTS Endpoint : {TTS_ENDPOINT}")
print(f"✅ Region       : {VOICE_SPEECH_REGION}")
print(f"✅ Voice Name   : {MAI_VOICE_NAME}")

## 2. Basic TTS — REST API

The simplest way to generate speech: POST SSML, receive audio bytes.

In [ ]:
def synthesize_to_file(
    text: str,
    output_path: str,
    voice: str = MAI_VOICE_NAME,
    output_format: str = "audio-24khz-160kbitrate-mono-mp3",
    lang: str = "en-US",
) -> float:
    """Synthesize text to audio using MAI-Voice-1 REST API. Returns elapsed time."""
    ssml = f"""<speak version='1.0' xml:lang='{lang}'>
  <voice xml:lang='{lang}' name='{voice}'>{text}</voice>
</speak>"""

    headers = {
        "Ocp-Apim-Subscription-Key": SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": output_format,
        "User-Agent": "MAI-Voice-Demo",
    }

    start = time.time()
    response = requests.post(TTS_ENDPOINT, headers=headers, data=ssml.encode("utf-8"))
    elapsed = time.time() - start

    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    size_kb = len(response.content) / 1024
    print(f"✅ Saved: {output_path} ({size_kb:.1f} KB)  |  API latency: {elapsed:.2f}s")
    return elapsed


TEXT_BASIC = (
    "Welcome to Microsoft Foundry. MAI-Voice-1 is a high-fidelity speech "
    "generation model that produces natural, expressive audio. "
    "It powers Copilot Audio Expressions and is now available for developers."
)

synthesize_to_file(TEXT_BASIC, "output_basic.mp3")

In [ ]:
# Inline playback in Jupyter
from IPython.display import Audio, display
display(Audio("output_basic.mp3"))

## 3. SSML with Emotion / Tone Control

MAI-Voice-1 supports per-turn emotion control through SSML `<mstts:express-as>` tags.

In [ ]:
def synthesize_ssml_to_file(ssml: str, output_path: str) -> None:
    """Send raw SSML directly to the TTS endpoint."""
    headers = {
        "Ocp-Apim-Subscription-Key": SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "audio-24khz-160kbitrate-mono-mp3",
        "User-Agent": "MAI-Voice-Demo",
    }
    response = requests.post(TTS_ENDPOINT, headers=headers, data=ssml.encode("utf-8"))
    response.raise_for_status()
    with open(output_path, "wb") as f:
        f.write(response.content)
    print(f"✅ Saved: {output_path}")


# Example: Express-as emotion tags for conversational styles
# Available styles depend on voice — check Azure Speech Studio for MAI-Voice-1 styles
SSML_EMOTION = f"""<speak version='1.0' xmlns='http://www.w3.org/2001/10/synthesis'
                         xmlns:mstts='http://www.w3.org/2001/mstts'
                         xml:lang='en-US'>
  <voice name='{MAI_VOICE_NAME}'>
    <mstts:express-as style='excited'>
      We're thrilled to announce MAI-Voice-1 is now available on Microsoft Foundry!
    </mstts:express-as>
    <break time='500ms'/>
    <mstts:express-as style='calm'>
      It delivers 60 seconds of audio in under one second on a single GPU.
    </mstts:express-as>
    <break time='300ms'/>
    <mstts:express-as style='empathetic'>
      Building great voice experiences has never been easier for developers.
    </mstts:express-as>
  </voice>
</speak>"""

synthesize_ssml_to_file(SSML_EMOTION, "output_emotion.mp3")
display(Audio("output_emotion.mp3"))

## 4. SSML Advanced: Prosody Control (Rate, Pitch, Volume)

In [ ]:
SSML_PROSODY = f"""<speak version='1.0' xml:lang='en-US'>
  <voice xml:lang='en-US' name='{MAI_VOICE_NAME}'>
    <!-- Slow down for emphasis -->
    <prosody rate='slow' pitch='+5%'>
      The price is twenty-two dollars per one million characters.
    </prosody>
    <break time='400ms'/>
    <!-- Speed up for a fast-paced segment -->
    <prosody rate='fast' volume='loud'>
      Deploy it today on Microsoft Foundry!
    </prosody>
  </voice>
</speak>"""

synthesize_ssml_to_file(SSML_PROSODY, "output_prosody.mp3")
display(Audio("output_prosody.mp3"))

## 5. Azure Speech SDK — Synthesize to Speaker

Using the Python SDK for real-time or SDK-based integration.

In [ ]:
def synthesize_with_sdk(text: str, output_file: str = "output_sdk.wav") -> None:
    """Synthesize text using the Azure Speech SDK (saves to file)."""
    speech_config = speechsdk.SpeechConfig(
        subscription=SPEECH_KEY,
        region=SPEECH_REGION
    )
    speech_config.speech_synthesis_voice_name = MAI_VOICE_NAME
    speech_config.set_speech_synthesis_output_format(
        speechsdk.SpeechSynthesisOutputFormat.Audio24Khz160KBitRateMonoMp3
    )

    audio_config = speechsdk.audio.AudioOutputConfig(filename=output_file)
    synthesizer  = speechsdk.SpeechSynthesizer(
        speech_config=speech_config,
        audio_config=audio_config
    )

    result = synthesizer.speak_text_async(text).get()

    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        size_kb = Path(output_file).stat().st_size / 1024
        print(f"✅ Synthesized {len(text)} chars → {output_file} ({size_kb:.1f} KB)")
    elif result.reason == speechsdk.ResultReason.Canceled:
        details = result.cancellation_details
        print(f"❌ Synthesis canceled: {details.reason}")
        if details.error_details:
            print(f"   Error: {details.error_details}")


synthesize_with_sdk(
    "MAI Voice One is brought to you by Microsoft AI, powering the next generation "
    "of voice experiences in Copilot and beyond."
)
display(Audio("output_sdk.wav"))

## 6. Voice Prompting (Concept Demo)

MAI-Voice-1 supports **voice prompting** — provide a short audio clip and the model clones that speaker's voice without fine-tuning.

> ⚠️ **Requires Microsoft approval** — custom voice creation is subject to [Microsoft's Responsible AI policies](https://learn.microsoft.com/legal/ai-code-of-conduct). The feature is available through **Azure Speech Personal Voice**.

In [ ]:
# Voice prompting is available through the Personal Voice API in Azure Speech.
# Reference: https://learn.microsoft.com/azure/ai-services/speech-service/personal-voice-overview

# Conceptual example — actual Personal Voice API requires approved access.
# The SSML reference voice tag is used to specify the cloned voice:

SSML_VOICE_PROMPT = f"""<speak version='1.0' xml:lang='en-US'>
  <voice name='{MAI_VOICE_NAME}'>
    <!--
    For voice prompting / Personal Voice:
    1. Obtain an approved Personal Voice speaker profile ID from Azure Speech.
    2. Use the DragonLatestNeural voice with the speaker profile ID.

    Example after approval:
    <voice name='DragonLatestNeural'>
      <mstts:ttsembedding speakerProfileId='your-speaker-profile-id'/>
      Your synthesized text here.
    </voice>
    -->
    This demonstrates MAI-Voice-1's voice prompting capability description.
    With an approved speaker profile, the model clones any voice from a
    ten-second audio sample — no fine-tuning required.
  </voice>
</speak>"""

print("Voice prompting SSML template shown above.")
print("See: https://learn.microsoft.com/azure/ai-services/speech-service/personal-voice-overview")

## 7. Long-Form Narration (Batch Synthesis)

For content >5 minutes (audiobooks, podcasts, lectures), use the Batch Synthesis API.

In [ ]:
LONG_TEXT = """Chapter One: The Dawn of Microsoft AI.

Microsoft AI has been developing world-class models that power millions of experiences 
across Copilot, Bing, PowerPoint, and Azure. Today marks a significant milestone as 
MAI-Voice-1, MAI-Transcribe-1, and MAI-Image-2 become available to every developer 
through Microsoft Foundry.

Chapter Two: Voice at Scale.

MAI-Voice-1 can generate sixty seconds of expressive, human-like audio in under one 
second on a single GPU. This makes real-time voice assistants, interactive IVR systems, 
and large-scale audiobook production economically viable for developers of all sizes.

Chapter Three: Building the Future.

With pricing starting at twenty-two dollars per million characters, developers can 
build rich voice experiences without breaking their budget. The future of human-computer 
interaction is voice — and MAI-Voice-1 is at the center of that future."""

print(f"Long-form text length: {len(LONG_TEXT)} characters")

# Synthesize using the SDK (suitable for texts up to a few thousand characters)
synthesize_with_sdk(LONG_TEXT, "output_longform.wav")
display(Audio("output_longform.wav"))

## 8. 💰 Cost Calculator

**MAI-Voice-1 pricing: \$22 per 1M characters**

Characters include spaces and punctuation (all characters in the input text).

In [ ]:
# ── Cost Calculator ─────────────────────────────────────────
PRICE_PER_1M_CHARS = 22.00  # USD per 1 million characters (as of April 2026)

# Average reading speeds:
#   ~ 500 chars/min (slow, conversational)
#   ~ 800 chars/min (natural speech)
#   ~ 1200 chars/min (fast narration)

scenarios = {
    "Single blog post (3,000 chars)":          3_000,
    "Short audiobook chapter (30,000 chars)":  30_000,
    "Full audiobook (~500K chars)":            500_000,
    "Daily IVR traffic (1M chars/day)":      1_000_000,
    "Enterprise monthly (50M chars)": 50_000_000,
}

print(f"\nMAI-Voice-1 Cost Estimator  (${PRICE_PER_1M_CHARS:.2f} / 1M chars)")
print(f"{'Scenario':<45} {'Characters':>12} {'Cost (USD)':>12}")
print("-" * 71)
for label, chars in scenarios.items():
    cost = (chars / 1_000_000) * PRICE_PER_1M_CHARS
    print(f"{label:<45} {chars:>12,} ${cost:>11.4f}")

# Current text from notebook
print()
print("Current demo cost:")
demo_chars = len(TEXT_BASIC) + len(LONG_TEXT)
demo_cost  = (demo_chars / 1_000_000) * PRICE_PER_1M_CHARS
print(f"  {demo_chars:,} chars → ${demo_cost:.6f}")

## 9. Summary & Next Steps

| Feature | Status |
|---|---|
| Basic TTS (REST) | ✅ |
| SSML emotion control | ✅ |
| SSML prosody (rate/pitch) | ✅ |
| Azure Speech SDK | ✅ |
| Voice prompting (Personal Voice) | ✅ (requires approval) |
| Long-form narration | ✅ |
| 10+ languages | 🔜 Coming soon |

**Resources:**
- [Model Card](https://ai.azure.com/catalog/models/MAI-Voice-1)
- [Azure Speech TTS docs](https://learn.microsoft.com/azure/ai-services/speech-service/text-to-speech)
- [SSML reference](https://learn.microsoft.com/azure/ai-services/speech-service/speech-synthesis-markup)
- [Personal Voice (voice cloning)](https://learn.microsoft.com/azure/ai-services/speech-service/personal-voice-overview)
- [MAI Playground](https://playground.microsoft.ai)